# Preparando os arquivos

Biblioteca do yolo

In [ ]:
!ls /content/RodoSol_degradada/placas_cortadas_ocr/

!cat /content/RodoSol_degradada/placas_cortadas_ocr/split_novo.txt


In [ ]:
# dependencias
%pip install ultralytics

import torch
import ultralytics

ultralytics.checks()

print(f"GPU disponível: {torch.cuda.is_available()}")

Puxando arquivo do drive

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import zipfile
from google.colab import drive
# monta drive e extrai datasets

if not os.path.exists('/content/RodoSol_degradada'):

  # monta drive

  drive.mount('/content/drive')
#tem batch 8 ou 16, cada um pra um fine tuning
  zip_path = "<DRIVE>/datasets/rodosol_dmt_8B.zip"   # caminho zip

  destino = "/content/RodoSol_degradada"

  # overwrite se ja tem pasta
  !unzip -oq {zip_path} -d {destino}

  # remover a pasta __MACOSX
  !rm -rf {destino}/__MACOSX

if not os.path.exists('/content/base_competicao'):

  drive.mount('/content/drive')

  zip_path = "<DRIVE>/datasets/LRLPR-26-5opEvJTW.zip"   # caminho zip

  destino = "/content/base_competicao"

  # overwrite se ja tem pasta
  !unzip -oq {zip_path} -d {destino}

  # remover a pasta __MACOSX
  !rm -rf {destino}/__MACOSX


# Atividade 5 - fine tuning de parseq tiny

Preparando arquivos

In [ ]:
import os
import json
import random
import shutil
from collections import defaultdict
import cv2

# ─── VARIAVEIS ────────────────────────────────────────────────────────────────
SPLIT_JSON_PATH    = '/content/split-train-val.json'
IMAGES_BASE_DIR    = '/content/base_competicao/LRLPR-26-5opEvJTW/train'
PARSEQ_DATASET_DIR = 'parseq_dataset'
SAMPLE_FRACTION    = 1.0
# ─────────────────────────────────────────────────────────────────────────────

if os.path.exists(PARSEQ_DATASET_DIR):
    shutil.rmtree(PARSEQ_DATASET_DIR)

os.makedirs(os.path.join(PARSEQ_DATASET_DIR, 'images/train'), exist_ok=True)
os.makedirs(os.path.join(PARSEQ_DATASET_DIR, 'images/val'),   exist_ok=True)

with open(SPLIT_JSON_PATH, 'r') as f:
    split_data = json.load(f)

dataset_files = defaultdict(list)
for scenario, splits in split_data.items():
    for split_name, track_paths in splits.items():
        for track_path_str in track_paths:
            dataset_files[split_name].append(track_path_str)

print("Tracks encontradas por conjunto:")
for split, tracks in dataset_files.items():
    print(f"  {split}: {len(tracks)} tracks")

parseq_labels = defaultdict(list)

for split, tracks in dataset_files.items():
    num_samples = int(len(tracks) * SAMPLE_FRACTION)
    sampled = random.sample(tracks, num_samples)

    print(f"\nProcessando {len(sampled)}/{len(tracks)} tracks para '{split}'...")

    for track_path_str in sampled:
        track_path = os.path.join(IMAGES_BASE_DIR, track_path_str.strip('/'))

        if not os.path.isdir(track_path):
            print(f"  aviso: pasta nao encontrada: {track_path}")
            continue

        annotation_file = os.path.join(track_path, 'annotations.json')
        if not os.path.exists(annotation_file):
            print(f"  aviso: annotations.json nao encontrado em {track_path}")
            continue

        with open(annotation_file, 'r') as af:
            ann = json.load(af)

        transcription_str = ann.get('plate_text')
        if not transcription_str:
            print(f"  aviso: sem plate_text em {annotation_file}")
            continue

        frames = sorted([f for f in os.listdir(track_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        if not frames:
            print(f"  aviso: sem imagens em {track_path}")
            continue

        track_id = track_path_str.strip('/').replace('/', '_')
        for i, frame in enumerate(frames):
            full_img_path = os.path.join(track_path, frame)
            image = cv2.imread(full_img_path)
            if image is None:
                continue
            crop_name = f"{track_id}_{i}_crop.png"
            crop_path = os.path.join(PARSEQ_DATASET_DIR, 'images', split, crop_name)
            cv2.imwrite(crop_path, image)
            parseq_labels[split].append((crop_name, transcription_str))

print("\nSalvando labels...")
for split, labels in parseq_labels.items():
    label_file = os.path.join(PARSEQ_DATASET_DIR, f'{split}_labels.txt')
    with open(label_file, 'w') as f:
        for img_name, text in labels:
            f.write(f"{img_name},{text.replace(' ', '')}\n")

print(f"\nDataset pronto em: '{PARSEQ_DATASET_DIR}'")
for split, labels in parseq_labels.items():
    print(f"  {split}: {len(labels)} amostras")

In [ ]:
import shutil

DRIVE_DATASET_PATH = '<DRIVE>/backups/parseq_dataset'

print("Salvando dataset no Drive...")
shutil.copytree('parseq_dataset', DRIVE_DATASET_PATH)
print(f"Dataset salvo em: {DRIVE_DATASET_PATH}")

Para fine tuning

In [ ]:
%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
import os
import pandas as pd

PARSEQ_DATASET_DIR = 'parseq_dataset'
TRAIN_IMG_DIR = os.path.join(PARSEQ_DATASET_DIR, 'images/train')
TRAIN_LBL_FILE = os.path.join(PARSEQ_DATASET_DIR, 'train_labels.txt')
VAL_IMG_DIR = os.path.join(PARSEQ_DATASET_DIR, 'images/val')
VAL_LBL_FILE = os.path.join(PARSEQ_DATASET_DIR, 'val_labels.txt')

# hiperparametros
BATCH_SIZE = 64
LEARNING_RATE = 1e-5
MAX_EPOCHS = 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# carrega parseq tiny pre-treinado
print("Carregando PARSeq Tiny pre-treinado...")
parseq_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=True).to(DEVICE)

parseq_system.train()

tokenizer = parseq_system.tokenizer

img_size = parseq_system.hparams.img_size
print(f"PARSeq carregado. img_size: {img_size}")

img_transform = T.Compose([
    T.Resize(img_size, T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

class OCRDataset(Dataset):
    def __init__(self, images_dir, labels_file, transform):
        self.images_dir = images_dir
        self.transform = transform

        try:
            self.data = pd.read_csv(labels_file, header=None, names=['img_name', 'text'], sep=',')

            self.data.dropna(subset=['text'], inplace=True)
            self.data['text'] = self.data['text'].astype(str)
            print(f"Carregado {len(self.data)} amostras de {labels_file}")
        except pd.errors.EmptyDataError:
            print(f"aviso: labels vazio: {labels_file}")
            self.data = pd.DataFrame(columns=['img_name', 'text'])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = row['img_name']
        text = row['text']

        img_path = os.path.join(self.images_dir, img_name)

        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            print(f"erro: imagem nao encontrada: {img_path}")

            image = Image.new('RGB', (img_size[1], img_size[0]), color='black')
            text = "ERROR"

        if self.transform:
            image = self.transform(image)

        return image, text

def collate_fn_parseq(batch):
    images, texts = zip(*batch)
    images = torch.stack(images, 0)

    return images, texts

print("\nCriando DataLoaders...")
train_dataset = OCRDataset(TRAIN_IMG_DIR, TRAIN_LBL_FILE, img_transform)
val_dataset = OCRDataset(VAL_IMG_DIR, VAL_LBL_FILE, img_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn_parseq
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn_parseq
)

print(f"DataLoaders: {len(train_dataset)} treino, {len(val_dataset)} validacao")

parseq_system.hparams.lr = LEARNING_RATE

from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='<DRIVE>/checkpoints/parseq_base',
    filename='parseq_tiny-{epoch:02d}-{val_loss:.4f}',
    save_top_k=3,
    monitor='val_loss',
    mode='min',
    save_last=True,
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator=DEVICE,
    log_every_n_steps=10,
    callbacks=[
        pl.callbacks.EarlyStopping(monitor='val_loss', patience=3, mode='min'),
        checkpoint_callback,
    ]
)

# fine-tuning
print(f"--- Iniciando Fine-Tuning em {DEVICE} por {MAX_EPOCHS} épocas ---")

CHECKPOINT_PATH = '<DRIVE>/checkpoints/parseq_base/last.ckpt'

if os.path.exists(CHECKPOINT_PATH):
    print(f"Retomando treino do checkpoint: {CHECKPOINT_PATH}")
    trainer.fit(parseq_system, train_loader, val_loader, ckpt_path=CHECKPOINT_PATH)
else:
    print("Nenhum checkpoint encontrado, iniciando do zero.")
    trainer.fit(parseq_system, train_loader, val_loader)

print("fine-tuning concluido.")

# salva pesos do modelo
MODEL_SAVE_PATH = "<DRIVE>/pesos/parseq_tiny_base.pt"
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

torch.save(parseq_system.model.state_dict(), MODEL_SAVE_PATH)

print(f"pesos salvos em: {MODEL_SAVE_PATH}")

Carrega tudo e testa

In [ ]:

%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import os
from ultralytics import YOLO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

parseq_finetuned_path = "<DRIVE>/pesos/parseq_tiny_base.pt"

print("carregando PARSeq Tiny...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# aplica pesos fine-tuned
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:

        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("pesos fine-tuned aplicados.")
    except Exception as e:
        print(f"erro ao carregar state_dict: {e}")
else:
    print(f"erro: pesos nao encontrados: {parseq_finetuned_path}")

recognizer_system.eval()
print("reconhecedor pronto.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3

img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")

for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  aviso: sem anotacao em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)

    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')

    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)
        predicted_text = recognizer_system.tokenizer.decode(logits.cpu())[0]

        results_by_category[category].append({
            'predicted': predicted_text,
            'ground_truth': gt_plate,
            'track': track_name,
            'frame': img_name
        })

print("reconhecimento concluido.")

In [ ]:
import os
from collections import defaultdict

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'

imgs_por_categoria = defaultdict(int)
tracks_por_categoria = defaultdict(int)

for track_name in os.listdir(test_base):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if os.path.exists(annotation_path):
        import json
        with open(annotation_path) as f:
            ann = json.load(f)
        categoria = ann.get('plate_layout', 'unknown')
    else:
        categoria = 'unknown'

    imgs = [f for f in os.listdir(track_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    imgs_por_categoria[categoria] += len(imgs)
    tracks_por_categoria[categoria] += 1

print("─── Imagens por categoria ───")
for cat in sorted(imgs_por_categoria):
    print(f"  {cat}: {imgs_por_categoria[cat]} imagens em {tracks_por_categoria[cat]} tracks")
print(f"\n  Total: {sum(imgs_por_categoria.values())} imagens em {sum(tracks_por_categoria.values())} tracks")

Análise de resultado

In [ ]:
# dic para a contagem de acertos e totais por categoria
correct_counts = defaultdict(int)
total_counts = defaultdict(int)

# itera sobre resultado em cada categoria
for category, results in results_by_category.items():
    if not results:
        continue

    for res in results:
        total_counts[category] += 1
        # limpando carcteres q nao sao letras ou numeros -> ocorrencia de carros BR com '-'
        res_predicted = "".join(caractere for caractere in res['predicted'] if caractere.isalnum())
        if res_predicted == res['ground_truth']:
            correct_counts[category] += 1

---------------------------------------------------------------------
print("\n--- Taxa de Reconhecimento de Placas (Accuracy) ---")

# para calc global
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

 (cat e global)
for category in sorted(total_counts.keys()):
    accuracy = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0
    print(f"- Categoria '{category}': {accuracy:.2f}% ({correct_counts[category]}/{total_counts[category]} placas corretas)")

global_accuracy = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0
print("-----------------------------------------------------")
print(f"Taxa de Reconhecimento GLOBAL: {global_accuracy:.2f}% ({total_correct_global}/{total_plates_global} placas corretas)")
print("-----------------------------------------------------")

# Fine tuning de parseq tiny com base de competição pura (resultados por track) ✅ 74.17%

Preparando arquivos (imgs fine tuning)

In [ ]:
import os
import json
import random
import shutil
from collections import defaultdict
import cv2

# ─── VARIAVEIS ────────────────────────────────────────────────────────────────
SPLIT_JSON_PATH    = '/content/split-train-val.json'
IMAGES_BASE_DIR    = '/content/base_competicao/LRLPR-26-5opEvJTW/train'
PARSEQ_DATASET_DIR = 'parseq_dataset'
SAMPLE_FRACTION    = 1.0
# ─────────────────────────────────────────────────────────────────────────────

if os.path.exists(PARSEQ_DATASET_DIR):
    shutil.rmtree(PARSEQ_DATASET_DIR)

os.makedirs(os.path.join(PARSEQ_DATASET_DIR, 'images/train'), exist_ok=True)
os.makedirs(os.path.join(PARSEQ_DATASET_DIR, 'images/val'),   exist_ok=True)

with open(SPLIT_JSON_PATH, 'r') as f:
    split_data = json.load(f)

dataset_files = defaultdict(list)
for scenario, splits in split_data.items():
    for split_name, track_paths in splits.items():
        for track_path_str in track_paths:
            dataset_files[split_name].append(track_path_str)

print("Tracks encontradas por conjunto:")
for split, tracks in dataset_files.items():
    print(f"  {split}: {len(tracks)} tracks")

parseq_labels = defaultdict(list)

for split, tracks in dataset_files.items():
    num_samples = int(len(tracks) * SAMPLE_FRACTION)
    sampled = random.sample(tracks, num_samples)

    print(f"\nProcessando {len(sampled)}/{len(tracks)} tracks para '{split}'...")

    for track_path_str in sampled:
        track_path = os.path.join(IMAGES_BASE_DIR, track_path_str.strip('/'))

        if not os.path.isdir(track_path):
            print(f"  aviso: pasta nao encontrada: {track_path}")
            continue

        annotation_file = os.path.join(track_path, 'annotations.json')
        if not os.path.exists(annotation_file):
            print(f"  aviso: annotations.json nao encontrado em {track_path}")
            continue

        with open(annotation_file, 'r') as af:
            ann = json.load(af)

        transcription_str = ann.get('plate_text')
        if not transcription_str:
            print(f"  aviso: sem plate_text em {annotation_file}")
            continue

        frames = sorted([f for f in os.listdir(track_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        if not frames:
            print(f"  aviso: sem imagens em {track_path}")
            continue

        track_id = track_path_str.strip('/').replace('/', '_')
        for i, frame in enumerate(frames):
            full_img_path = os.path.join(track_path, frame)
            image = cv2.imread(full_img_path)
            if image is None:
                continue
            crop_name = f"{track_id}_{i}_crop.png"
            crop_path = os.path.join(PARSEQ_DATASET_DIR, 'images', split, crop_name)
            cv2.imwrite(crop_path, image)
            parseq_labels[split].append((crop_name, transcription_str))

print("\nSalvando labels...")
for split, labels in parseq_labels.items():
    label_file = os.path.join(PARSEQ_DATASET_DIR, f'{split}_labels.txt')
    with open(label_file, 'w') as f:
        for img_name, text in labels:
            f.write(f"{img_name},{text.replace(' ', '')}\n")

print(f"\nDataset pronto em: '{PARSEQ_DATASET_DIR}'")
for split, labels in parseq_labels.items():
    print(f"  {split}: {len(labels)} amostras")

In [ ]:
import shutil

DRIVE_DATASET_PATH = '<DRIVE>/backups/parseq_dataset'

print("Salvando dataset no Drive...")
shutil.copytree('parseq_dataset', DRIVE_DATASET_PATH)
print(f"Dataset salvo em: {DRIVE_DATASET_PATH}")

Para fine tuning

In [ ]:
%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
import os
import pandas as pd

PARSEQ_DATASET_DIR = 'parseq_dataset'
TRAIN_IMG_DIR = os.path.join(PARSEQ_DATASET_DIR, 'images/train')
TRAIN_LBL_FILE = os.path.join(PARSEQ_DATASET_DIR, 'train_labels.txt')
VAL_IMG_DIR = os.path.join(PARSEQ_DATASET_DIR, 'images/val')
VAL_LBL_FILE = os.path.join(PARSEQ_DATASET_DIR, 'val_labels.txt')

# hiperparametros
BATCH_SIZE = 64
LEARNING_RATE = 1e-5
MAX_EPOCHS = 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# carrega parseq tiny pre-treinado
print("Carregando PARSeq Tiny pre-treinado...")
parseq_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=True).to(DEVICE)

parseq_system.train()

tokenizer = parseq_system.tokenizer

img_size = parseq_system.hparams.img_size
print(f"PARSeq carregado. img_size: {img_size}")

img_transform = T.Compose([
    T.Resize(img_size, T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

class OCRDataset(Dataset):
    def __init__(self, images_dir, labels_file, transform):
        self.images_dir = images_dir
        self.transform = transform

        try:
            self.data = pd.read_csv(labels_file, header=None, names=['img_name', 'text'], sep=',')

            self.data.dropna(subset=['text'], inplace=True)
            self.data['text'] = self.data['text'].astype(str)
            print(f"Carregado {len(self.data)} amostras de {labels_file}")
        except pd.errors.EmptyDataError:
            print(f"aviso: labels vazio: {labels_file}")
            self.data = pd.DataFrame(columns=['img_name', 'text'])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = row['img_name']
        text = row['text']

        img_path = os.path.join(self.images_dir, img_name)

        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            print(f"erro: imagem nao encontrada: {img_path}")

            image = Image.new('RGB', (img_size[1], img_size[0]), color='black')
            text = "ERROR"

        if self.transform:
            image = self.transform(image)

        return image, text

def collate_fn_parseq(batch):
    images, texts = zip(*batch)
    images = torch.stack(images, 0)

    return images, texts

print("\nCriando DataLoaders...")
train_dataset = OCRDataset(TRAIN_IMG_DIR, TRAIN_LBL_FILE, img_transform)
val_dataset = OCRDataset(VAL_IMG_DIR, VAL_LBL_FILE, img_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn_parseq
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn_parseq
)

print(f"DataLoaders: {len(train_dataset)} treino, {len(val_dataset)} validacao")

parseq_system.hparams.lr = LEARNING_RATE

from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='<DRIVE>/checkpoints/parseq_base',
    filename='parseq_tiny-{epoch:02d}-{val_loss:.4f}',
    save_top_k=3,
    monitor='val_loss',
    mode='min',
    save_last=True,
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator=DEVICE,
    log_every_n_steps=10,
    callbacks=[
        pl.callbacks.EarlyStopping(monitor='val_loss', patience=3, mode='min'),
        checkpoint_callback,
    ]
)

# fine-tuning
print(f"--- Iniciando Fine-Tuning em {DEVICE} por {MAX_EPOCHS} épocas ---")

CHECKPOINT_PATH = '<DRIVE>/checkpoints/parseq_base/last.ckpt'

if os.path.exists(CHECKPOINT_PATH):
    print(f"Retomando treino do checkpoint: {CHECKPOINT_PATH}")
    trainer.fit(parseq_system, train_loader, val_loader, ckpt_path=CHECKPOINT_PATH)
else:
    print("Nenhum checkpoint encontrado, iniciando do zero.")
    trainer.fit(parseq_system, train_loader, val_loader)

print("fine-tuning concluido.")

# salva pesos do modelo
MODEL_SAVE_PATH = "<DRIVE>/pesos/parseq_tiny_base.pt"
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

torch.save(parseq_system.model.state_dict(), MODEL_SAVE_PATH)

print(f"pesos salvos em: {MODEL_SAVE_PATH}")

Carrega tudo e testa

In [ ]:

%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import os
from ultralytics import YOLO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

parseq_finetuned_path = "<DRIVE>/pesos/parseq_tiny_base.pt"

print("carregando PARSeq Tiny...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# aplica pesos fine-tuned
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:

        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("pesos fine-tuned aplicados.")
    except Exception as e:
        print(f"erro ao carregar state_dict: {e}")
else:
    print(f"erro: pesos nao encontrados: {parseq_finetuned_path}")

recognizer_system.eval()
print("reconhecedor pronto.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  aviso: sem anotacao em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predicoes e scores de todos os frames da track
    track_predictions = []

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva predicoes da track para MVCP
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions
    })

print("reconhecimento concluido.")

In [ ]:
import os
from collections import defaultdict

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'

imgs_por_categoria = defaultdict(int)
tracks_por_categoria = defaultdict(int)

for track_name in os.listdir(test_base):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if os.path.exists(annotation_path):
        import json
        with open(annotation_path) as f:
            ann = json.load(f)
        categoria = ann.get('plate_layout', 'unknown')
    else:
        categoria = 'unknown'

    imgs = [f for f in os.listdir(track_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    imgs_por_categoria[categoria] += len(imgs)
    tracks_por_categoria[categoria] += 1

print("─── Imagens por categoria ───")
for cat in sorted(imgs_por_categoria):
    print(f"  {cat}: {imgs_por_categoria[cat]} imagens em {tracks_por_categoria[cat]} tracks")
print(f"\n  Total: {sum(imgs_por_categoria.values())} imagens em {sum(tracks_por_categoria.values())} tracks")

Análise de resultado

In [ ]:
from collections import defaultdict, Counter

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # remove caracteres nao alfanumericos
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # comprimento mais comum entre as predicoes
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra predicoes com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confianca) de cada frame nessa posicao
        char_scores = []
        for text, scores in clean:
            char = text[pos]

            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate pela media de confianca
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)

correct_counts = defaultdict(int)
total_counts = defaultdict(int)

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        if prediction == res['ground_truth']:
            correct_counts[category] += 1

print("\n--- Recognition Rate por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0
    print(f"- Categoria '{category}': {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas)")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0
print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print("------------------------------------------")

# Fine tuning de parseq tiny com base de competição mista batch 8 TSIT (resultados por track) ❌ 73.63%

Preparando arquivos (imgs fine tuning)

In [ ]:
import os
import json
import random
import shutil
from collections import defaultdict
import cv2

# ─── VARIAVEIS ────────────────────────────────────────────────────────────────
SPLIT_JSON_PATH    = '/content/split-train-val.json'
IMAGES_BASE_DIR    = '/content/base_competicao/LRLPR-26-5opEvJTW/train'
PARSEQ_DATASET_DIR = 'parseq_dataset_misto'
SAMPLE_FRACTION    = 1.0
# ─────────────────────────────────────────────────────────────────────────────

if os.path.exists(PARSEQ_DATASET_DIR):
    shutil.rmtree(PARSEQ_DATASET_DIR)

os.makedirs(os.path.join(PARSEQ_DATASET_DIR, 'images/train'), exist_ok=True)
os.makedirs(os.path.join(PARSEQ_DATASET_DIR, 'images/val'),   exist_ok=True)

with open(SPLIT_JSON_PATH, 'r') as f:
    split_data = json.load(f)

dataset_files = defaultdict(list)
for scenario, splits in split_data.items():
    for split_name, track_paths in splits.items():
        for track_path_str in track_paths:
            dataset_files[split_name].append(track_path_str)

print("Tracks encontradas por conjunto:")
for split, tracks in dataset_files.items():
    print(f"  {split}: {len(tracks)} tracks")

parseq_labels = defaultdict(list)

for split, tracks in dataset_files.items():
    num_samples = int(len(tracks) * SAMPLE_FRACTION)
    sampled = random.sample(tracks, num_samples)

    print(f"\nProcessando {len(sampled)}/{len(tracks)} tracks para '{split}'...")

    for track_path_str in sampled:
        track_path = os.path.join(IMAGES_BASE_DIR, track_path_str.strip('/'))

        if not os.path.isdir(track_path):
            print(f"  aviso: pasta nao encontrada: {track_path}")
            continue

        annotation_file = os.path.join(track_path, 'annotations.json')
        if not os.path.exists(annotation_file):
            print(f"  aviso: annotations.json nao encontrado em {track_path}")
            continue

        with open(annotation_file, 'r') as af:
            ann = json.load(af)

        transcription_str = ann.get('plate_text')
        if not transcription_str:
            print(f"  aviso: sem plate_text em {annotation_file}")
            continue

        frames = sorted([f for f in os.listdir(track_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        if not frames:
            print(f"  aviso: sem imagens em {track_path}")
            continue

        track_id = track_path_str.strip('/').replace('/', '_')
        for i, frame in enumerate(frames):
            full_img_path = os.path.join(track_path, frame)
            image = cv2.imread(full_img_path)
            if image is None:
                continue
            crop_name = f"{track_id}_{i}_crop.png"
            crop_path = os.path.join(PARSEQ_DATASET_DIR, 'images', split, crop_name)
            cv2.imwrite(crop_path, image)
            parseq_labels[split].append((crop_name, transcription_str))

print("\nSalvando labels...")
for split, labels in parseq_labels.items():
    label_file = os.path.join(PARSEQ_DATASET_DIR, f'{split}_labels.txt')
    with open(label_file, 'w') as f:
        for img_name, text in labels:
            f.write(f"{img_name},{text.replace(' ', '')}\n")

print(f"\nDataset pronto em: '{PARSEQ_DATASET_DIR}'")
for split, labels in parseq_labels.items():
    print(f"  {split}: {len(labels)} amostras")

In [ ]:
import os
import cv2

RODOSOL_DIR    = '/content/RodoSol_degradada/placas_cortadas_ocr'
SPLIT_FILE     = os.path.join(RODOSOL_DIR, 'split_novo.txt')

# mapeia split_novo.txt para os mesmos nomes de split usados no parseq_dataset
SPLIT_MAP = {
    'training':   'train',
    'validation': 'train',
    'testing':    'train',  # testing da RodoSol vai pro treino também
}

print("Adicionando dados da RodoSol degradada ao parseq_dataset...")

# le o split
split_entries = {}
with open(SPLIT_FILE, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        img_name, split_name = line.split(';')
        split_entries[img_name] = split_name

added = defaultdict(int)

for img_name, split_name in split_entries.items():
    split_dest = SPLIT_MAP.get(split_name)
    if split_dest is None:
        continue

    img_name_disk = os.path.splitext(img_name)[0] + '.png'
    img_path = os.path.join(RODOSOL_DIR, img_name_disk)
    txt_path  = os.path.join(RODOSOL_DIR, os.path.splitext(img_name)[0] + '.txt')

    if not os.path.exists(img_path) or not os.path.exists(txt_path):
        continue

    # le o txt para pegar a placa
    plate_text = None
    with open(txt_path, 'r') as f:
        for line in f:
            if line.startswith('plate:'):
                plate_text = line.split(':', 1)[1].strip()
                break

    if not plate_text:
        continue

    # copia imagem para o destino
    image = cv2.imread(img_path)
    if image is None:
        continue

    crop_name = f"rodosol_{os.path.splitext(img_name)[0]}_crop.png"
    dest_path = os.path.join(PARSEQ_DATASET_DIR, 'images', split_dest, crop_name)
    cv2.imwrite(dest_path, image)

    # adiciona label
    parseq_labels[split_dest].append((crop_name, plate_text))
    added[split_dest] += 1

# regrava labels
print("Regravando arquivos de label...")
for split, labels in parseq_labels.items():
    label_file = os.path.join(PARSEQ_DATASET_DIR, f'{split}_labels.txt')
    with open(label_file, 'w') as f:
        for img_name, text in labels:
            f.write(f"{img_name},{text.replace(' ', '')}\n")

print("RodoSol degradada adicionada:")
for split, count in sorted(added.items()):
    print(f"  {split}: +{count} amostras")
print(f"\nTotais finais:")
for split, labels in parseq_labels.items():
    print(f"  {split}: {len(labels)} amostras")

In [ ]:
print(f"Total de entradas no split: {len(split_entries)}")
print(f"Exemplo: {list(split_entries.items())[:3]}")
print(f"PARSEQ_DATASET_DIR: {PARSEQ_DATASET_DIR}")
print(f"Pasta destino train existe: {os.path.exists(os.path.join(PARSEQ_DATASET_DIR, 'images', 'train'))}")
print(f"Pasta RODOSOL existe: {os.path.exists(RODOSOL_DIR)}")

In [ ]:
import shutil

DRIVE_DATASET_PATH = '<DRIVE>/backups/parseq_dataset_misto'

print("Salvando dataset no Drive...")
shutil.copytree('parseq_dataset_misto', DRIVE_DATASET_PATH)
print(f"Dataset salvo em: {DRIVE_DATASET_PATH}")

Para fine tuning

In [ ]:
%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
import os
import pandas as pd

PARSEQ_DATASET_DIR = 'parseq_dataset_misto'
TRAIN_IMG_DIR = os.path.join(PARSEQ_DATASET_DIR, 'images/train')
TRAIN_LBL_FILE = os.path.join(PARSEQ_DATASET_DIR, 'train_labels.txt')
VAL_IMG_DIR = os.path.join(PARSEQ_DATASET_DIR, 'images/val')
VAL_LBL_FILE = os.path.join(PARSEQ_DATASET_DIR, 'val_labels.txt')

# hiperparametros
BATCH_SIZE = 64
LEARNING_RATE = 1e-5
MAX_EPOCHS = 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# carrega parseq tiny pre-treinado
print("Carregando PARSeq Tiny pre-treinado...")
parseq_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=True).to(DEVICE)

parseq_system.train()

tokenizer = parseq_system.tokenizer

img_size = parseq_system.hparams.img_size
print(f"PARSeq carregado. img_size: {img_size}")

img_transform = T.Compose([
    T.Resize(img_size, T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

class OCRDataset(Dataset):
    def __init__(self, images_dir, labels_file, transform):
        self.images_dir = images_dir
        self.transform = transform

        try:
            self.data = pd.read_csv(labels_file, header=None, names=['img_name', 'text'], sep=',')

            self.data.dropna(subset=['text'], inplace=True)
            self.data['text'] = self.data['text'].astype(str)
            print(f"Carregado {len(self.data)} amostras de {labels_file}")
        except pd.errors.EmptyDataError:
            print(f"aviso: labels vazio: {labels_file}")
            self.data = pd.DataFrame(columns=['img_name', 'text'])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = row['img_name']
        text = row['text']

        img_path = os.path.join(self.images_dir, img_name)

        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            print(f"erro: imagem nao encontrada: {img_path}")

            image = Image.new('RGB', (img_size[1], img_size[0]), color='black')
            text = "ERROR"

        if self.transform:
            image = self.transform(image)

        return image, text

def collate_fn_parseq(batch):
    images, texts = zip(*batch)
    images = torch.stack(images, 0)

    return images, texts

print("\nCriando DataLoaders...")
train_dataset = OCRDataset(TRAIN_IMG_DIR, TRAIN_LBL_FILE, img_transform)
val_dataset = OCRDataset(VAL_IMG_DIR, VAL_LBL_FILE, img_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn_parseq
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn_parseq
)

print(f"DataLoaders: {len(train_dataset)} treino, {len(val_dataset)} validacao")

parseq_system.hparams.lr = LEARNING_RATE

from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='<DRIVE>/checkpoints/parseq_misto_tsit8',
    filename='parseq_tiny-{epoch:02d}-{val_loss:.4f}',
    save_top_k=3,
    monitor='val_loss',
    mode='min',
    save_last=True,
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator=DEVICE,
    log_every_n_steps=10,
    callbacks=[
        pl.callbacks.EarlyStopping(monitor='val_loss', patience=3, mode='min'),
        checkpoint_callback,
    ]
)

# fine-tuning
print(f"--- Iniciando Fine-Tuning em {DEVICE} por {MAX_EPOCHS} épocas ---")

CHECKPOINT_PATH = '<DRIVE>/checkpoints/parseq_misto_tsit8/last.ckpt'

if os.path.exists(CHECKPOINT_PATH):
    print(f"Retomando treino do checkpoint: {CHECKPOINT_PATH}")
    trainer.fit(parseq_system, train_loader, val_loader, ckpt_path=CHECKPOINT_PATH)
else:
    print("Nenhum checkpoint encontrado, iniciando do zero.")
    trainer.fit(parseq_system, train_loader, val_loader)

print("fine-tuning concluido.")

# salva pesos do modelo
MODEL_SAVE_PATH = "<DRIVE>/pesos/parseq_tiny_tsit8.pt"
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

torch.save(parseq_system.model.state_dict(), MODEL_SAVE_PATH)

print(f"pesos salvos em: {MODEL_SAVE_PATH}")

Carrega tudo e testa

In [ ]:

%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import os
from ultralytics import YOLO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

parseq_finetuned_path = "<DRIVE>/pesos/parseq_tiny_tsit8.pt"

print("carregando PARSeq Tiny...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# aplica pesos fine-tuned
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:

        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("pesos fine-tuned aplicados.")
    except Exception as e:
        print(f"erro ao carregar state_dict: {e}")
else:
    print(f"erro: pesos nao encontrados: {parseq_finetuned_path}")

recognizer_system.eval()
print("reconhecedor pronto.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  aviso: sem anotacao em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predicoes e scores de todos os frames da track
    track_predictions = []

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva predicoes da track para MVCP
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions
    })

print("reconhecimento concluido.")

In [ ]:
import os
from collections import defaultdict

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'

imgs_por_categoria = defaultdict(int)
tracks_por_categoria = defaultdict(int)

for track_name in os.listdir(test_base):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if os.path.exists(annotation_path):
        import json
        with open(annotation_path) as f:
            ann = json.load(f)
        categoria = ann.get('plate_layout', 'unknown')
    else:
        categoria = 'unknown'

    imgs = [f for f in os.listdir(track_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    imgs_por_categoria[categoria] += len(imgs)
    tracks_por_categoria[categoria] += 1

print("─── Imagens por categoria ───")
for cat in sorted(imgs_por_categoria):
    print(f"  {cat}: {imgs_por_categoria[cat]} imagens em {tracks_por_categoria[cat]} tracks")
print(f"\n  Total: {sum(imgs_por_categoria.values())} imagens em {sum(tracks_por_categoria.values())} tracks")

Análise de resultado

In [ ]:
from collections import defaultdict, Counter

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # remove caracteres nao alfanumericos
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # comprimento mais comum entre as predicoes
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra predicoes com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confianca) de cada frame nessa posicao
        char_scores = []
        for text, scores in clean:
            char = text[pos]

            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate pela media de confianca
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)

correct_counts = defaultdict(int)
total_counts = defaultdict(int)

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        if prediction == res['ground_truth']:
            correct_counts[category] += 1

print("\n--- Recognition Rate por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0
    print(f"- Categoria '{category}': {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas)")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0
print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print("------------------------------------------")

# Fine tuning de parseq tiny com base de competição mista batch 16 TSIT (resultados por track) ✅ 74.93%

Preparando arquivos (imgs fine tuning)

In [ ]:
import os
import json
import random
import shutil
from collections import defaultdict
import cv2

# ─── VARIAVEIS ────────────────────────────────────────────────────────────────
SPLIT_JSON_PATH    = '/content/split-train-val.json'
IMAGES_BASE_DIR    = '/content/base_competicao/LRLPR-26-5opEvJTW/train'
PARSEQ_DATASET_DIR = 'parseq_dataset_misto_16'
SAMPLE_FRACTION    = 1.0
# ─────────────────────────────────────────────────────────────────────────────

if os.path.exists(PARSEQ_DATASET_DIR):
    shutil.rmtree(PARSEQ_DATASET_DIR)

os.makedirs(os.path.join(PARSEQ_DATASET_DIR, 'images/train'), exist_ok=True)
os.makedirs(os.path.join(PARSEQ_DATASET_DIR, 'images/val'),   exist_ok=True)

with open(SPLIT_JSON_PATH, 'r') as f:
    split_data = json.load(f)

dataset_files = defaultdict(list)
for scenario, splits in split_data.items():
    for split_name, track_paths in splits.items():
        for track_path_str in track_paths:
            dataset_files[split_name].append(track_path_str)

print("Tracks encontradas por conjunto:")
for split, tracks in dataset_files.items():
    print(f"  {split}: {len(tracks)} tracks")

parseq_labels = defaultdict(list)

for split, tracks in dataset_files.items():
    num_samples = int(len(tracks) * SAMPLE_FRACTION)
    sampled = random.sample(tracks, num_samples)

    print(f"\nProcessando {len(sampled)}/{len(tracks)} tracks para '{split}'...")

    for track_path_str in sampled:
        track_path = os.path.join(IMAGES_BASE_DIR, track_path_str.strip('/'))

        if not os.path.isdir(track_path):
            print(f"  aviso: pasta nao encontrada: {track_path}")
            continue

        annotation_file = os.path.join(track_path, 'annotations.json')
        if not os.path.exists(annotation_file):
            print(f"  aviso: annotations.json nao encontrado em {track_path}")
            continue

        with open(annotation_file, 'r') as af:
            ann = json.load(af)

        transcription_str = ann.get('plate_text')
        if not transcription_str:
            print(f"  aviso: sem plate_text em {annotation_file}")
            continue

        frames = sorted([f for f in os.listdir(track_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        if not frames:
            print(f"  aviso: sem imagens em {track_path}")
            continue

        track_id = track_path_str.strip('/').replace('/', '_')
        for i, frame in enumerate(frames):
            full_img_path = os.path.join(track_path, frame)
            image = cv2.imread(full_img_path)
            if image is None:
                continue
            crop_name = f"{track_id}_{i}_crop.png"
            crop_path = os.path.join(PARSEQ_DATASET_DIR, 'images', split, crop_name)
            cv2.imwrite(crop_path, image)
            parseq_labels[split].append((crop_name, transcription_str))

print("\nSalvando labels...")
for split, labels in parseq_labels.items():
    label_file = os.path.join(PARSEQ_DATASET_DIR, f'{split}_labels.txt')
    with open(label_file, 'w') as f:
        for img_name, text in labels:
            f.write(f"{img_name},{text.replace(' ', '')}\n")

print(f"\nDataset pronto em: '{PARSEQ_DATASET_DIR}'")
for split, labels in parseq_labels.items():
    print(f"  {split}: {len(labels)} amostras")

In [ ]:
import os
import cv2

RODOSOL_DIR    = '/content/RodoSol_degradada/placas_cortadas_ocr'
SPLIT_FILE     = os.path.join(RODOSOL_DIR, 'split_novo.txt')

# mapeia split_novo.txt para os mesmos nomes de split usados no parseq_dataset
SPLIT_MAP = {
    'training':   'train',
    'validation': 'train',
    'testing':    'train',  # testing da RodoSol vai pro treino também
}

print("Adicionando dados da RodoSol degradada ao parseq_dataset...")

# le o split
split_entries = {}
with open(SPLIT_FILE, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        img_name, split_name = line.split(';')
        split_entries[img_name] = split_name

added = defaultdict(int)

for img_name, split_name in split_entries.items():
    split_dest = SPLIT_MAP.get(split_name)
    if split_dest is None:
        continue

    img_name_disk = os.path.splitext(img_name)[0] + '.png'
    img_path = os.path.join(RODOSOL_DIR, img_name_disk)
    txt_path  = os.path.join(RODOSOL_DIR, os.path.splitext(img_name)[0] + '.txt')

    if not os.path.exists(img_path) or not os.path.exists(txt_path):
        continue

    # le o txt para pegar a placa
    plate_text = None
    with open(txt_path, 'r') as f:
        for line in f:
            if line.startswith('plate:'):
                plate_text = line.split(':', 1)[1].strip()
                break

    if not plate_text:
        continue

    # copia imagem para o destino
    image = cv2.imread(img_path)
    if image is None:
        continue

    crop_name = f"rodosol_{os.path.splitext(img_name)[0]}_crop.png"
    dest_path = os.path.join(PARSEQ_DATASET_DIR, 'images', split_dest, crop_name)
    cv2.imwrite(dest_path, image)

    # adiciona label
    parseq_labels[split_dest].append((crop_name, plate_text))
    added[split_dest] += 1

# regrava labels
print("Regravando arquivos de label...")
for split, labels in parseq_labels.items():
    label_file = os.path.join(PARSEQ_DATASET_DIR, f'{split}_labels.txt')
    with open(label_file, 'w') as f:
        for img_name, text in labels:
            f.write(f"{img_name},{text.replace(' ', '')}\n")

print("RodoSol degradada adicionada:")
for split, count in sorted(added.items()):
    print(f"  {split}: +{count} amostras")
print(f"\nTotais finais:")
for split, labels in parseq_labels.items():
    print(f"  {split}: {len(labels)} amostras")

In [ ]:
print(f"Total de entradas no split: {len(split_entries)}")
print(f"Exemplo: {list(split_entries.items())[:3]}")
print(f"PARSEQ_DATASET_DIR: {PARSEQ_DATASET_DIR}")
print(f"Pasta destino train existe: {os.path.exists(os.path.join(PARSEQ_DATASET_DIR, 'images', 'train'))}")
print(f"Pasta RODOSOL existe: {os.path.exists(RODOSOL_DIR)}")

In [ ]:
import shutil

DRIVE_DATASET_PATH = '<DRIVE>/backups/parseq_dataset_misto2'

print("Salvando dataset no Drive...")
shutil.copytree('parseq_dataset_misto', DRIVE_DATASET_PATH)
print(f"Dataset salvo em: {DRIVE_DATASET_PATH}")

Para fine tuning

In [ ]:
%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
import os
import pandas as pd

PARSEQ_DATASET_DIR = 'parseq_dataset_misto_16'
TRAIN_IMG_DIR = os.path.join(PARSEQ_DATASET_DIR, 'images/train')
TRAIN_LBL_FILE = os.path.join(PARSEQ_DATASET_DIR, 'train_labels.txt')
VAL_IMG_DIR = os.path.join(PARSEQ_DATASET_DIR, 'images/val')
VAL_LBL_FILE = os.path.join(PARSEQ_DATASET_DIR, 'val_labels.txt')

# hiperparametros
BATCH_SIZE = 64
LEARNING_RATE = 1e-5
MAX_EPOCHS = 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# carrega parseq tiny pre-treinado
print("Carregando PARSeq Tiny pre-treinado...")
parseq_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=True).to(DEVICE)

parseq_system.train()

tokenizer = parseq_system.tokenizer

img_size = parseq_system.hparams.img_size
print(f"PARSeq carregado. img_size: {img_size}")

img_transform = T.Compose([
    T.Resize(img_size, T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

class OCRDataset(Dataset):
    def __init__(self, images_dir, labels_file, transform):
        self.images_dir = images_dir
        self.transform = transform

        try:
            self.data = pd.read_csv(labels_file, header=None, names=['img_name', 'text'], sep=',')

            self.data.dropna(subset=['text'], inplace=True)
            self.data['text'] = self.data['text'].astype(str)
            print(f"Carregado {len(self.data)} amostras de {labels_file}")
        except pd.errors.EmptyDataError:
            print(f"aviso: labels vazio: {labels_file}")
            self.data = pd.DataFrame(columns=['img_name', 'text'])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = row['img_name']
        text = row['text']

        img_path = os.path.join(self.images_dir, img_name)

        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            print(f"erro: imagem nao encontrada: {img_path}")

            image = Image.new('RGB', (img_size[1], img_size[0]), color='black')
            text = "ERROR"

        if self.transform:
            image = self.transform(image)

        return image, text

def collate_fn_parseq(batch):
    images, texts = zip(*batch)
    images = torch.stack(images, 0)

    return images, texts

print("\nCriando DataLoaders...")
train_dataset = OCRDataset(TRAIN_IMG_DIR, TRAIN_LBL_FILE, img_transform)
val_dataset = OCRDataset(VAL_IMG_DIR, VAL_LBL_FILE, img_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn_parseq
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn_parseq
)

print(f"DataLoaders: {len(train_dataset)} treino, {len(val_dataset)} validacao")

parseq_system.hparams.lr = LEARNING_RATE

from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='<DRIVE>/checkpoints/parseq_misto_tsit16',
    filename='parseq_tiny-{epoch:02d}-{val_loss:.4f}',
    save_top_k=3,
    monitor='val_loss',
    mode='min',
    save_last=True,
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator=DEVICE,
    log_every_n_steps=10,
    callbacks=[
        pl.callbacks.EarlyStopping(monitor='val_loss', patience=3, mode='min'),
        checkpoint_callback,
    ]
)

# fine-tuning
print(f"--- Iniciando Fine-Tuning em {DEVICE} por {MAX_EPOCHS} épocas ---")

CHECKPOINT_PATH = '<DRIVE>/checkpoints/parseq_misto_tsit16/last.ckpt'

if os.path.exists(CHECKPOINT_PATH):
    print(f"Retomando treino do checkpoint: {CHECKPOINT_PATH}")
    trainer.fit(parseq_system, train_loader, val_loader, ckpt_path=CHECKPOINT_PATH)
else:
    print("Nenhum checkpoint encontrado, iniciando do zero.")
    trainer.fit(parseq_system, train_loader, val_loader)

print("fine-tuning concluido.")

# salva pesos do modelo
MODEL_SAVE_PATH = "<DRIVE>/pesos/parseq_tiny_tsit16.pt"
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

torch.save(parseq_system.model.state_dict(), MODEL_SAVE_PATH)

print(f"pesos salvos em: {MODEL_SAVE_PATH}")

Carrega tudo e testa

In [ ]:

%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import os
from ultralytics import YOLO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

parseq_finetuned_path = "<DRIVE>/pesos/parseq_tiny_tsit16.pt"

print("carregando PARSeq Tiny...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# aplica pesos fine-tuned
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:

        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("pesos fine-tuned aplicados.")
    except Exception as e:
        print(f"erro ao carregar state_dict: {e}")
else:
    print(f"erro: pesos nao encontrados: {parseq_finetuned_path}")

recognizer_system.eval()
print("reconhecedor pronto.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  aviso: sem anotacao em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predicoes e scores de todos os frames da track
    track_predictions = []

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva predicoes da track para MVCP
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions
    })

print("reconhecimento concluido.")

In [ ]:
import os
from collections import defaultdict

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'

imgs_por_categoria = defaultdict(int)
tracks_por_categoria = defaultdict(int)

for track_name in os.listdir(test_base):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if os.path.exists(annotation_path):
        import json
        with open(annotation_path) as f:
            ann = json.load(f)
        categoria = ann.get('plate_layout', 'unknown')
    else:
        categoria = 'unknown'

    imgs = [f for f in os.listdir(track_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    imgs_por_categoria[categoria] += len(imgs)
    tracks_por_categoria[categoria] += 1

print("─── Imagens por categoria ───")
for cat in sorted(imgs_por_categoria):
    print(f"  {cat}: {imgs_por_categoria[cat]} imagens em {tracks_por_categoria[cat]} tracks")
print(f"\n  Total: {sum(imgs_por_categoria.values())} imagens em {sum(tracks_por_categoria.values())} tracks")

Análise de resultado

In [ ]:
from collections import defaultdict, Counter

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # remove caracteres nao alfanumericos
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # comprimento mais comum entre as predicoes
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra predicoes com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confianca) de cada frame nessa posicao
        char_scores = []
        for text, scores in clean:
            char = text[pos]

            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate pela media de confianca
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)

correct_counts = defaultdict(int)
total_counts = defaultdict(int)

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        if prediction == res['ground_truth']:
            correct_counts[category] += 1

print("\n--- Recognition Rate por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0
    print(f"- Categoria '{category}': {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas)")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0
print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print("------------------------------------------")

# Fine tuning de parseq tiny com base de competição mista batch 16 DMT (resultados por track) ✅ 74.33%

Preparando arquivos (imgs fine tuning)

In [ ]:
import os
import json
import random
import shutil
from collections import defaultdict
import cv2

# ─── VARIAVEIS ────────────────────────────────────────────────────────────────
SPLIT_JSON_PATH    = '<DRIVE>/datasets/split-train-val.json'
IMAGES_BASE_DIR    = '/content/base_competicao/LRLPR-26-5opEvJTW/train'
PARSEQ_DATASET_DIR = 'parseq_dataset_misto_16'
SAMPLE_FRACTION    = 1.0
# ─────────────────────────────────────────────────────────────────────────────

if os.path.exists(PARSEQ_DATASET_DIR):
    shutil.rmtree(PARSEQ_DATASET_DIR)

os.makedirs(os.path.join(PARSEQ_DATASET_DIR, 'images/train'), exist_ok=True)
os.makedirs(os.path.join(PARSEQ_DATASET_DIR, 'images/val'),   exist_ok=True)

with open(SPLIT_JSON_PATH, 'r') as f:
    split_data = json.load(f)

dataset_files = defaultdict(list)
for scenario, splits in split_data.items():
    for split_name, track_paths in splits.items():
        for track_path_str in track_paths:
            dataset_files[split_name].append(track_path_str)

print("Tracks encontradas por conjunto:")
for split, tracks in dataset_files.items():
    print(f"  {split}: {len(tracks)} tracks")

parseq_labels = defaultdict(list)

for split, tracks in dataset_files.items():
    num_samples = int(len(tracks) * SAMPLE_FRACTION)
    sampled = random.sample(tracks, num_samples)

    print(f"\nProcessando {len(sampled)}/{len(tracks)} tracks para '{split}'...")

    for track_path_str in sampled:
        track_path = os.path.join(IMAGES_BASE_DIR, track_path_str.strip('/'))

        if not os.path.isdir(track_path):
            print(f"  aviso: pasta nao encontrada: {track_path}")
            continue

        annotation_file = os.path.join(track_path, 'annotations.json')
        if not os.path.exists(annotation_file):
            print(f"  aviso: annotations.json nao encontrado em {track_path}")
            continue

        with open(annotation_file, 'r') as af:
            ann = json.load(af)

        transcription_str = ann.get('plate_text')
        if not transcription_str:
            print(f"  aviso: sem plate_text em {annotation_file}")
            continue

        frames = sorted([f for f in os.listdir(track_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        if not frames:
            print(f"  aviso: sem imagens em {track_path}")
            continue

        track_id = track_path_str.strip('/').replace('/', '_')
        for i, frame in enumerate(frames):
            full_img_path = os.path.join(track_path, frame)
            image = cv2.imread(full_img_path)
            if image is None:
                continue
            crop_name = f"{track_id}_{i}_crop.png"
            crop_path = os.path.join(PARSEQ_DATASET_DIR, 'images', split, crop_name)
            cv2.imwrite(crop_path, image)
            parseq_labels[split].append((crop_name, transcription_str))

print("\nSalvando labels...")
for split, labels in parseq_labels.items():
    label_file = os.path.join(PARSEQ_DATASET_DIR, f'{split}_labels.txt')
    with open(label_file, 'w') as f:
        for img_name, text in labels:
            f.write(f"{img_name},{text.replace(' ', '')}\n")

print(f"\nDataset pronto em: '{PARSEQ_DATASET_DIR}'")
for split, labels in parseq_labels.items():
    print(f"  {split}: {len(labels)} amostras")

In [ ]:
import os
import cv2

RODOSOL_DIR    = '/content/RodoSol_degradada/placas_cortadas_ocr'
SPLIT_FILE     = os.path.join(RODOSOL_DIR, 'split_novo.txt')

# mapeia split_novo.txt para os mesmos nomes de split usados no parseq_dataset
SPLIT_MAP = {
    'training':   'train',
    'validation': 'train',
    'testing':    'train',  # testing da RodoSol vai pro treino também
}

print("Adicionando dados da RodoSol degradada ao parseq_dataset...")

# le o split
split_entries = {}
with open(SPLIT_FILE, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        img_name, split_name = line.split(';')
        split_entries[img_name] = split_name

added = defaultdict(int)

for img_name, split_name in split_entries.items():
    split_dest = SPLIT_MAP.get(split_name)
    if split_dest is None:
        continue

    img_name_disk = os.path.splitext(img_name)[0] + '.png'
    img_path = os.path.join(RODOSOL_DIR, img_name_disk)
    txt_path  = os.path.join(RODOSOL_DIR, os.path.splitext(img_name)[0] + '.txt')

    if not os.path.exists(img_path) or not os.path.exists(txt_path):
        continue

    # le o txt para pegar a placa
    plate_text = None
    with open(txt_path, 'r') as f:
        for line in f:
            if line.startswith('plate:'):
                plate_text = line.split(':', 1)[1].strip()
                break

    if not plate_text:
        continue

    # copia imagem para o destino
    image = cv2.imread(img_path)
    if image is None:
        continue

    crop_name = f"rodosol_{os.path.splitext(img_name)[0]}_crop.png"
    dest_path = os.path.join(PARSEQ_DATASET_DIR, 'images', split_dest, crop_name)
    cv2.imwrite(dest_path, image)

    # adiciona label
    parseq_labels[split_dest].append((crop_name, plate_text))
    added[split_dest] += 1

# regrava labels
print("Regravando arquivos de label...")
for split, labels in parseq_labels.items():
    label_file = os.path.join(PARSEQ_DATASET_DIR, f'{split}_labels.txt')
    with open(label_file, 'w') as f:
        for img_name, text in labels:
            f.write(f"{img_name},{text.replace(' ', '')}\n")

print("RodoSol degradada adicionada:")
for split, count in sorted(added.items()):
    print(f"  {split}: +{count} amostras")
print(f"\nTotais finais:")
for split, labels in parseq_labels.items():
    print(f"  {split}: {len(labels)} amostras")

In [ ]:
print(f"Total de entradas no split: {len(split_entries)}")
print(f"Exemplo: {list(split_entries.items())[:3]}")
print(f"PARSEQ_DATASET_DIR: {PARSEQ_DATASET_DIR}")
print(f"Pasta destino train existe: {os.path.exists(os.path.join(PARSEQ_DATASET_DIR, 'images', 'train'))}")
print(f"Pasta RODOSOL existe: {os.path.exists(RODOSOL_DIR)}")

In [ ]:
# (backup de dataset removido)

Para fine tuning

In [ ]:
%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
import os
import pandas as pd

PARSEQ_DATASET_DIR = 'parseq_dataset_misto_16'
TRAIN_IMG_DIR = os.path.join(PARSEQ_DATASET_DIR, 'images/train')
TRAIN_LBL_FILE = os.path.join(PARSEQ_DATASET_DIR, 'train_labels.txt')
VAL_IMG_DIR = os.path.join(PARSEQ_DATASET_DIR, 'images/val')
VAL_LBL_FILE = os.path.join(PARSEQ_DATASET_DIR, 'val_labels.txt')

# hiperparametros
BATCH_SIZE = 64
LEARNING_RATE = 1e-5
MAX_EPOCHS = 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# carrega parseq tiny pre-treinado
print("Carregando PARSeq Tiny pre-treinado...")
parseq_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=True).to(DEVICE)

parseq_system.train()

tokenizer = parseq_system.tokenizer

img_size = parseq_system.hparams.img_size
print(f"PARSeq carregado. img_size: {img_size}")

img_transform = T.Compose([
    T.Resize(img_size, T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

class OCRDataset(Dataset):
    def __init__(self, images_dir, labels_file, transform):
        self.images_dir = images_dir
        self.transform = transform

        try:
            self.data = pd.read_csv(labels_file, header=None, names=['img_name', 'text'], sep=',')

            self.data.dropna(subset=['text'], inplace=True)
            self.data['text'] = self.data['text'].astype(str)
            print(f"Carregado {len(self.data)} amostras de {labels_file}")
        except pd.errors.EmptyDataError:
            print(f"aviso: labels vazio: {labels_file}")
            self.data = pd.DataFrame(columns=['img_name', 'text'])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = row['img_name']
        text = row['text']

        img_path = os.path.join(self.images_dir, img_name)

        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            print(f"erro: imagem nao encontrada: {img_path}")

            image = Image.new('RGB', (img_size[1], img_size[0]), color='black')
            text = "ERROR"

        if self.transform:
            image = self.transform(image)

        return image, text

def collate_fn_parseq(batch):
    images, texts = zip(*batch)
    images = torch.stack(images, 0)

    return images, texts

print("\nCriando DataLoaders...")
train_dataset = OCRDataset(TRAIN_IMG_DIR, TRAIN_LBL_FILE, img_transform)
val_dataset = OCRDataset(VAL_IMG_DIR, VAL_LBL_FILE, img_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn_parseq
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn_parseq
)

print(f"DataLoaders: {len(train_dataset)} treino, {len(val_dataset)} validacao")

parseq_system.hparams.lr = LEARNING_RATE

from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='<DRIVE>/checkpoints/parseq_misto_dmt16',
    filename='parseq_tiny-{epoch:02d}-{val_loss:.4f}',
    save_top_k=3,
    monitor='val_loss',
    mode='min',
    save_last=True,
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator=DEVICE,
    log_every_n_steps=10,
    callbacks=[
        pl.callbacks.EarlyStopping(monitor='val_loss', patience=3, mode='min'),
        checkpoint_callback,
    ]
)

# fine-tuning
print(f"--- Iniciando Fine-Tuning em {DEVICE} por {MAX_EPOCHS} épocas ---")

CHECKPOINT_PATH = '<DRIVE>/checkpoints/parseq_misto_dmt16/last.ckpt'

if os.path.exists(CHECKPOINT_PATH):
    print(f"Retomando treino do checkpoint: {CHECKPOINT_PATH}")
    trainer.fit(parseq_system, train_loader, val_loader, ckpt_path=CHECKPOINT_PATH)
else:
    print("Nenhum checkpoint encontrado, iniciando do zero.")
    trainer.fit(parseq_system, train_loader, val_loader)

print("fine-tuning concluido.")

# salva pesos do modelo
MODEL_SAVE_PATH = "<DRIVE>/pesos/parseq_tiny_dmt16.pt"
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

torch.save(parseq_system.model.state_dict(), MODEL_SAVE_PATH)

print(f"pesos salvos em: {MODEL_SAVE_PATH}")

Carrega tudo e testa

In [ ]:

%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import os
from ultralytics import YOLO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

parseq_finetuned_path = "<DRIVE>/pesos/parseq_tiny_dmt16.pt"

print("carregando PARSeq Tiny...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# aplica pesos fine-tuned
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:

        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("pesos fine-tuned aplicados.")
    except Exception as e:
        print(f"erro ao carregar state_dict: {e}")
else:
    print(f"erro: pesos nao encontrados: {parseq_finetuned_path}")

recognizer_system.eval()
print("reconhecedor pronto.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  aviso: sem anotacao em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predicoes e scores de todos os frames da track
    track_predictions = []

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva predicoes da track para MVCP
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions
    })

print("reconhecimento concluido.")

In [ ]:
import os
from collections import defaultdict

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'

imgs_por_categoria = defaultdict(int)
tracks_por_categoria = defaultdict(int)

for track_name in os.listdir(test_base):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if os.path.exists(annotation_path):
        import json
        with open(annotation_path) as f:
            ann = json.load(f)
        categoria = ann.get('plate_layout', 'unknown')
    else:
        categoria = 'unknown'

    imgs = [f for f in os.listdir(track_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    imgs_por_categoria[categoria] += len(imgs)
    tracks_por_categoria[categoria] += 1

print("─── Imagens por categoria ───")
for cat in sorted(imgs_por_categoria):
    print(f"  {cat}: {imgs_por_categoria[cat]} imagens em {tracks_por_categoria[cat]} tracks")
print(f"\n  Total: {sum(imgs_por_categoria.values())} imagens em {sum(tracks_por_categoria.values())} tracks")

Análise de resultado

In [ ]:
from collections import defaultdict, Counter

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # remove caracteres nao alfanumericos
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # comprimento mais comum entre as predicoes
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra predicoes com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confianca) de cada frame nessa posicao
        char_scores = []
        for text, scores in clean:
            char = text[pos]

            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate pela media de confianca
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)

correct_counts = defaultdict(int)
total_counts = defaultdict(int)

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        if prediction == res['ground_truth']:
            correct_counts[category] += 1

print("\n--- Recognition Rate por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0
    print(f"- Categoria '{category}': {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas)")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0
print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print("------------------------------------------")

# Fine tuning de parseq tiny com base de competição mista batch 8 DMT (resultados por track) ✅ 74.37%

Preparando arquivos (imgs fine tuning)

In [ ]:
import os
import json
import random
import shutil
from collections import defaultdict
import cv2

# ─── VARIAVEIS ────────────────────────────────────────────────────────────────
SPLIT_JSON_PATH    = '<DRIVE>/datasets/split-train-val.json'
IMAGES_BASE_DIR    = '/content/base_competicao/LRLPR-26-5opEvJTW/train'
PARSEQ_DATASET_DIR = 'parseq_dataset_misto_8'
SAMPLE_FRACTION    = 1.0
# ─────────────────────────────────────────────────────────────────────────────

if os.path.exists(PARSEQ_DATASET_DIR):
    shutil.rmtree(PARSEQ_DATASET_DIR)

os.makedirs(os.path.join(PARSEQ_DATASET_DIR, 'images/train'), exist_ok=True)
os.makedirs(os.path.join(PARSEQ_DATASET_DIR, 'images/val'),   exist_ok=True)

with open(SPLIT_JSON_PATH, 'r') as f:
    split_data = json.load(f)

dataset_files = defaultdict(list)
for scenario, splits in split_data.items():
    for split_name, track_paths in splits.items():
        for track_path_str in track_paths:
            dataset_files[split_name].append(track_path_str)

print("Tracks encontradas por conjunto:")
for split, tracks in dataset_files.items():
    print(f"  {split}: {len(tracks)} tracks")

parseq_labels = defaultdict(list)

for split, tracks in dataset_files.items():
    num_samples = int(len(tracks) * SAMPLE_FRACTION)
    sampled = random.sample(tracks, num_samples)

    print(f"\nProcessando {len(sampled)}/{len(tracks)} tracks para '{split}'...")

    for track_path_str in sampled:
        track_path = os.path.join(IMAGES_BASE_DIR, track_path_str.strip('/'))

        if not os.path.isdir(track_path):
            print(f"  aviso: pasta nao encontrada: {track_path}")
            continue

        annotation_file = os.path.join(track_path, 'annotations.json')
        if not os.path.exists(annotation_file):
            print(f"  aviso: annotations.json nao encontrado em {track_path}")
            continue

        with open(annotation_file, 'r') as af:
            ann = json.load(af)

        transcription_str = ann.get('plate_text')
        if not transcription_str:
            print(f"  aviso: sem plate_text em {annotation_file}")
            continue

        frames = sorted([f for f in os.listdir(track_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        if not frames:
            print(f"  aviso: sem imagens em {track_path}")
            continue

        track_id = track_path_str.strip('/').replace('/', '_')
        for i, frame in enumerate(frames):
            full_img_path = os.path.join(track_path, frame)
            image = cv2.imread(full_img_path)
            if image is None:
                continue
            crop_name = f"{track_id}_{i}_crop.png"
            crop_path = os.path.join(PARSEQ_DATASET_DIR, 'images', split, crop_name)
            cv2.imwrite(crop_path, image)
            parseq_labels[split].append((crop_name, transcription_str))

print("\nSalvando labels...")
for split, labels in parseq_labels.items():
    label_file = os.path.join(PARSEQ_DATASET_DIR, f'{split}_labels.txt')
    with open(label_file, 'w') as f:
        for img_name, text in labels:
            f.write(f"{img_name},{text.replace(' ', '')}\n")

print(f"\nDataset pronto em: '{PARSEQ_DATASET_DIR}'")
for split, labels in parseq_labels.items():
    print(f"  {split}: {len(labels)} amostras")

In [ ]:
import os
import cv2

RODOSOL_DIR    = '/content/RodoSol_degradada/placas_cortadas_ocr'
SPLIT_FILE     = os.path.join(RODOSOL_DIR, 'split_novo.txt')

# mapeia split_novo.txt para os mesmos nomes de split usados no parseq_dataset
SPLIT_MAP = {
    'training':   'train',
    'validation': 'train',
    'testing':    'train',  # testing da RodoSol vai pro treino também
}

print("Adicionando dados da RodoSol degradada ao parseq_dataset...")

# le o split
split_entries = {}
with open(SPLIT_FILE, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        img_name, split_name = line.split(';')
        split_entries[img_name] = split_name

added = defaultdict(int)

for img_name, split_name in split_entries.items():
    split_dest = SPLIT_MAP.get(split_name)
    if split_dest is None:
        continue

    img_name_disk = os.path.splitext(img_name)[0] + '.png'
    img_path = os.path.join(RODOSOL_DIR, img_name_disk)
    txt_path  = os.path.join(RODOSOL_DIR, os.path.splitext(img_name)[0] + '.txt')

    if not os.path.exists(img_path) or not os.path.exists(txt_path):
        continue

    # le o txt para pegar a placa
    plate_text = None
    with open(txt_path, 'r') as f:
        for line in f:
            if line.startswith('plate:'):
                plate_text = line.split(':', 1)[1].strip()
                break

    if not plate_text:
        continue

    # copia imagem para o destino
    image = cv2.imread(img_path)
    if image is None:
        continue

    crop_name = f"rodosol_{os.path.splitext(img_name)[0]}_crop.png"
    dest_path = os.path.join(PARSEQ_DATASET_DIR, 'images', split_dest, crop_name)
    cv2.imwrite(dest_path, image)

    # adiciona label
    parseq_labels[split_dest].append((crop_name, plate_text))
    added[split_dest] += 1

# regrava labels
print("Regravando arquivos de label...")
for split, labels in parseq_labels.items():
    label_file = os.path.join(PARSEQ_DATASET_DIR, f'{split}_labels.txt')
    with open(label_file, 'w') as f:
        for img_name, text in labels:
            f.write(f"{img_name},{text.replace(' ', '')}\n")

print("RodoSol degradada adicionada:")
for split, count in sorted(added.items()):
    print(f"  {split}: +{count} amostras")
print(f"\nTotais finais:")
for split, labels in parseq_labels.items():
    print(f"  {split}: {len(labels)} amostras")

In [ ]:
print(f"Total de entradas no split: {len(split_entries)}")
print(f"Exemplo: {list(split_entries.items())[:3]}")
print(f"PARSEQ_DATASET_DIR: {PARSEQ_DATASET_DIR}")
print(f"Pasta destino train existe: {os.path.exists(os.path.join(PARSEQ_DATASET_DIR, 'images', 'train'))}")
print(f"Pasta RODOSOL existe: {os.path.exists(RODOSOL_DIR)}")

In [ ]:
# (backup de dataset removido)

Para fine tuning

In [ ]:
%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
import os
import pandas as pd

PARSEQ_DATASET_DIR = 'parseq_dataset_misto_8'
TRAIN_IMG_DIR = os.path.join(PARSEQ_DATASET_DIR, 'images/train')
TRAIN_LBL_FILE = os.path.join(PARSEQ_DATASET_DIR, 'train_labels.txt')
VAL_IMG_DIR = os.path.join(PARSEQ_DATASET_DIR, 'images/val')
VAL_LBL_FILE = os.path.join(PARSEQ_DATASET_DIR, 'val_labels.txt')

# hiperparametros
BATCH_SIZE = 64
LEARNING_RATE = 1e-5
MAX_EPOCHS = 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# carrega parseq tiny pre-treinado
print("Carregando PARSeq Tiny pre-treinado...")
parseq_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=True).to(DEVICE)

parseq_system.train()

tokenizer = parseq_system.tokenizer

img_size = parseq_system.hparams.img_size
print(f"PARSeq carregado. img_size: {img_size}")

img_transform = T.Compose([
    T.Resize(img_size, T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

class OCRDataset(Dataset):
    def __init__(self, images_dir, labels_file, transform):
        self.images_dir = images_dir
        self.transform = transform

        try:
            self.data = pd.read_csv(labels_file, header=None, names=['img_name', 'text'], sep=',')

            self.data.dropna(subset=['text'], inplace=True)
            self.data['text'] = self.data['text'].astype(str)
            print(f"Carregado {len(self.data)} amostras de {labels_file}")
        except pd.errors.EmptyDataError:
            print(f"aviso: labels vazio: {labels_file}")
            self.data = pd.DataFrame(columns=['img_name', 'text'])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = row['img_name']
        text = row['text']

        img_path = os.path.join(self.images_dir, img_name)

        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            print(f"erro: imagem nao encontrada: {img_path}")

            image = Image.new('RGB', (img_size[1], img_size[0]), color='black')
            text = "ERROR"

        if self.transform:
            image = self.transform(image)

        return image, text

def collate_fn_parseq(batch):
    images, texts = zip(*batch)
    images = torch.stack(images, 0)

    return images, texts

print("\nCriando DataLoaders...")
train_dataset = OCRDataset(TRAIN_IMG_DIR, TRAIN_LBL_FILE, img_transform)
val_dataset = OCRDataset(VAL_IMG_DIR, VAL_LBL_FILE, img_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn_parseq
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn_parseq
)

print(f"DataLoaders: {len(train_dataset)} treino, {len(val_dataset)} validacao")

parseq_system.hparams.lr = LEARNING_RATE

from pytorch_lightning.callbacks import ModelCheckpoint

checkpoint_callback = ModelCheckpoint(
    dirpath='<DRIVE>/checkpoints/parseq_misto_dmt8',
    filename='parseq_tiny-{epoch:02d}-{val_loss:.4f}',
    save_top_k=3,
    monitor='val_loss',
    mode='min',
    save_last=True,
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator=DEVICE,
    log_every_n_steps=10,
    callbacks=[
        pl.callbacks.EarlyStopping(monitor='val_loss', patience=3, mode='min'),
        checkpoint_callback,
    ]
)

# fine-tuning
print(f"--- Iniciando Fine-Tuning em {DEVICE} por {MAX_EPOCHS} épocas ---")

CHECKPOINT_PATH = '<DRIVE>/checkpoints/parseq_misto_dmt8/last.ckpt'

if os.path.exists(CHECKPOINT_PATH):
    print(f"Retomando treino do checkpoint: {CHECKPOINT_PATH}")
    trainer.fit(parseq_system, train_loader, val_loader, ckpt_path=CHECKPOINT_PATH)
else:
    print("Nenhum checkpoint encontrado, iniciando do zero.")
    trainer.fit(parseq_system, train_loader, val_loader)

print("fine-tuning concluido.")

# salva pesos do modelo
MODEL_SAVE_PATH = "<DRIVE>/pesos/parseq_tiny_dmt8.pt"
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

torch.save(parseq_system.model.state_dict(), MODEL_SAVE_PATH)

print(f"pesos salvos em: {MODEL_SAVE_PATH}")

Carrega tudo e testa

In [ ]:

%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning

import torch
import os
from ultralytics import YOLO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")

parseq_finetuned_path = "<DRIVE>/pesos/parseq_tiny_dmt8.pt"

print("carregando PARSeq Tiny...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# aplica pesos fine-tuned
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:

        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("pesos fine-tuned aplicados.")
    except Exception as e:
        print(f"erro ao carregar state_dict: {e}")
else:
    print(f"erro: pesos nao encontrados: {parseq_finetuned_path}")

recognizer_system.eval()
print("reconhecedor pronto.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  aviso: sem anotacao em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predicoes e scores de todos os frames da track
    track_predictions = []

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva predicoes da track para MVCP
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions
    })

print("reconhecimento concluido.")

In [ ]:
import os
from collections import defaultdict

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'

imgs_por_categoria = defaultdict(int)
tracks_por_categoria = defaultdict(int)

for track_name in os.listdir(test_base):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if os.path.exists(annotation_path):
        import json
        with open(annotation_path) as f:
            ann = json.load(f)
        categoria = ann.get('plate_layout', 'unknown')
    else:
        categoria = 'unknown'

    imgs = [f for f in os.listdir(track_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    imgs_por_categoria[categoria] += len(imgs)
    tracks_por_categoria[categoria] += 1

print("─── Imagens por categoria ───")
for cat in sorted(imgs_por_categoria):
    print(f"  {cat}: {imgs_por_categoria[cat]} imagens em {tracks_por_categoria[cat]} tracks")
print(f"\n  Total: {sum(imgs_por_categoria.values())} imagens em {sum(tracks_por_categoria.values())} tracks")

Análise de resultado

In [ ]:
from collections import defaultdict, Counter

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # remove caracteres nao alfanumericos
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # comprimento mais comum entre as predicoes
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra predicoes com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confianca) de cada frame nessa posicao
        char_scores = []
        for text, scores in clean:
            char = text[pos]

            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate pela media de confianca
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)

correct_counts = defaultdict(int)
total_counts = defaultdict(int)

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        if prediction == res['ground_truth']:
            correct_counts[category] += 1

print("\n--- Recognition Rate por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0
    print(f"- Categoria '{category}': {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas)")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0
print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print("------------------------------------------")